LoRA Finetuning method

In [ ]:
!nvidia-smi

Sun Aug  9 15:18:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import userdata
from urllib.parse import quote
import subprocess

username = userdata.get("GITLAB_USERNAME")
token = userdata.get("GITLAB_TOKEN")

url = (
    f"https://{quote(username, safe='')}:{quote(token, safe='')}"
    f"@git.cs.bham.ac.uk/txh517/msc_project.git"
)

subprocess.run(
    ["git", "clone", url, "/content/msc_project"],
    check=True,
)

print("Repository cloned successfully.")

Repository cloned successfully.


In [ ]:
%cd /content/msc_project
!ls

/content/msc_project
baselines  config.yaml	experiments	README.md	  tests
common	   data		pyproject.toml	requirements.txt


In [ ]:
!grep -n "peak_gpu_memory" experiments/train_lora.py

336:            "peak_gpu_memory_allocated_gb": peak_allocated_gb,
337:            "peak_gpu_memory_reserved_gb": peak_reserved_gb,


In [ ]:
%pip install -q transformers datasets peft trl accelerate
%pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.4 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/spider_data.zip")
data_dir = Path("/content/msc_project/data/spider")

data_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(data_dir)

print("Spider extracted.")

Spider extracted.


In [ ]:
from pathlib import Path
import shutil

source = Path("/content/msc_project/data/spider/spider_data")
target = Path("/content/msc_project/data/spider")

for item in source.iterdir():
    destination = target / item.name

    if item.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.move(str(item), str(destination))
    else:
        shutil.move(str(item), str(destination))

shutil.rmtree(source)

print("Spider folder fixed.")

Spider folder fixed.


In [ ]:
!find /content/msc_project/data/spider -maxdepth 2 \
    -type f \( -name "train_spider.json" -o -name "dev.json" -o -name "tables.json" \)

/content/msc_project/data/spider/train_spider.json
/content/msc_project/data/spider/tables.json
/content/msc_project/data/spider/dev.json


In [ ]:
from pathlib import Path

config_path = Path("/content/msc_project/config.yaml")

text = config_path.read_text()
text = text.replace("device: cpu", "device: cuda")
config_path.write_text(text)

print("Changed device to cuda.")

Changed device to cuda.


In [ ]:
!grep "device:" config.yaml

device: cuda           # cuda | cpu


In [ ]:
%cd /content/msc_project

!python3 -m experiments.train_lora \
    --output_dir "/content/drive/MyDrive/msc_text_to_sql/models/qwen_spider_lora_r32_e3" \
    --epochs 3 \
    --batch_size 1 \
    --gradient_accumulation 16 \
    --learning_rate 1e-4 \
    --max_length 2048 \
    --save_steps 100 \
    --lora_rank 32 \
    --lora_alpha 64 \
    --lora_dropout 0.05

/content/msc_project
Training examples: 7000
Base model: Qwen/Qwen2.5-Coder-3B-Instruct
GPU: NVIDIA A100-SXM4-40GB
config.json: 100% 661/661 [00:00<00:00, 3.19MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 110MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 108MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 149MB/s]
Token lengths: median=2, p95=2, max=2
Sequences over max_length=2048: 0/7000 (0.0%)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 88.2MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.96G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   5% 335M/6.17G [00:02<00:33, 172MB/s, 6.39MB/s  ]
Reconstructing (incomplete total...):  86% 5.33G/6.17G [00:14<00:02, 392MB/s,  173MB/s  ]
Reconstru

In [ ]:
from pathlib import Path

adapter = Path(
    "/content/drive/MyDrive/msc_text_to_sql/models/"
    "qwen_spider_lora_r32_e3/final_adapter"
)

print("Adapter exists:", adapter.exists())

for p in adapter.iterdir():
    print(p.name)

Adapter exists: True
README.md
adapter_model.safetensors
adapter_config.json
chat_template.jinja
tokenizer_config.json
tokenizer.json
training_args.bin
training_metadata.json


In [ ]:
import json

metadata_path = (
    "/content/drive/MyDrive/msc_text_to_sql/models/"
    "qwen_spider_lora_r32_e3/final_adapter/"
    "training_metadata.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("Training loss:",
      metadata["training"]["train_loss"])

print("Trainable parameters:",
      f'{metadata["efficiency"]["trainable_parameters"]:,}')

print("Trainable percentage:",
      f'{metadata["efficiency"]["trainable_percentage"]:.4f}%')

print("Peak allocated VRAM:",
      f'{metadata["efficiency"]["peak_gpu_memory_allocated_gb"]:.2f} GB')

print("Peak reserved VRAM:",
      f'{metadata["efficiency"]["peak_gpu_memory_reserved_gb"]:.2f} GB')

throughput = metadata["efficiency"]["training_tokens_per_second"]

if throughput is not None:
    print("Training throughput:",
          f"{throughput:.2f} tokens/s")

Training loss: 0.08825723862446117
Trainable parameters: 14,745,600
Trainable percentage: 0.4756%
Peak allocated VRAM: 8.14 GB
Peak reserved VRAM: 8.74 GB
Training throughput: 927.12 tokens/s


In [ ]:
%cd /content/msc_project

!python3 -m experiments.run_slm_lora \
    --adapter "/content/drive/MyDrive/msc_text_to_sql/models/qwen_spider_lora_r32_e3/final_adapter" \
    --output "/content/drive/MyDrive/msc_text_to_sql/results/slm_lora_r32_e3.json" \
    --save_every 10

/content/msc_project
Loading base model Qwen/Qwen2.5-Coder-3B-Instruct ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:01<00:00, 231.91it/s]
Evaluating 1034 examples (checkpointing every 10) ...
[10/1034] running EX=0.800 (0.76 ex/s, ~22.6 min remaining)
[20/1034] running EX=0.800 (0.80 ex/s, ~21.2 min remaining)
[30/1034] running EX=0.833 (0.62 ex/s, ~26.8 min remaining)
[40/1034] running EX=0.850 (0.51 ex/s, ~32.4 min remaining)
[50/1034] running EX=0.820 (0.48 ex/s, ~34.4 min remaining)
[60/1034] running EX=0.767 (0.43 ex/s, ~38.2 min remaining)
[70/1034] running EX=0.714 (0.38 ex/s, ~42.0 min remaining)
[80/1034] running EX=0.688 (0.38 ex/s, ~41.7 min remaining)
[90/1034] running EX=0.722 (0.38 ex/s, ~41.4 min remaining)
[100/1034] running EX=0.690 (0.39 ex/s, ~40.4 min remaining)
[110/1034] running EX=0.645 (0.38 ex/s, ~40.5 min remaining)
[120/1034] running EX=0.617 (0.39 ex/s, ~39.5 min remaining)
[130/1034] running EX=0.615

In [ ]:
import json

result_path = (
    "/content/drive/MyDrive/msc_text_to_sql/results/"
    "slm_lora_r32_e3.json"
)

with open(result_path, "r") as f:
    results = json.load(f)

correct = sum(bool(x["correct"]) for x in results)
total = len(results)
accuracy = correct / total

print(f"Examples: {total}")
print(f"Correct: {correct}")
print(f"Execution accuracy: {accuracy:.2%}")

Examples: 1034
Correct: 739
Execution accuracy: 71.47%
